
# DEU Triangular Foam — Flat $2+1$D SR Fingerprint Audit

This notebook is a falsification-oriented follow-up to the triangular-foam MM / spatial-dimension result.

The question here is narrower than GR:

> In the cap-balanced geometric shoulder, do DEU foam causal diamonds behave like flat $2+1$D Minkowski / special-relativistic intervals?

The target signatures are:

- Myrheim--Meyer dimension near $D=3$.
- Ordering fraction near $r(D=3) \approx 0.228571$.
- Proper-time/volume law $V \sim h^3$, where $h$ is longest-chain height.
- Spatial cross-section law $A_{mid}\sim h^2$ and/or $A_{max}\sim h^2$.
- Normalized layer profiles closer to $2+1$D Minkowski controls than to chain, $3+1$D Minkowski, or random-DAG controls.

This is **not** a GR or curvature claim. It is the flat-spacetime compatibility screen that should precede any GR effort.



## Pass/fail rule

The selected foam window is considered promising only if it passes both types of test:

1. **Scaling:** $MM\approx 3$, $V\sim h^3$, and $A\sim h^2$ over a usable height ladder.
2. **Fingerprint:** its normalized layer profile is closest to the $2+1$D Minkowski control among the controls generated here.

A failure is useful. If the foam has $MM\approx 3$ but its interval profile is closer to a chain, random poset, or $3+1$D control, then the previous MM shoulder is not enough to call it SR-like.


In [ ]:

from pathlib import Path
import os
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if (ROOT / "src").exists():
    BASE = ROOT
else:
    # Allows running the notebook from the notebooks/ directory.
    BASE = ROOT.parent if (ROOT.parent / "src").exists() else ROOT

OUT = BASE / "outputs"
FIG = BASE / "figures"
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
print("BASE =", BASE)
print("OUT  =", OUT)
print("FIG  =", FIG)



## Load the DEU engine and SR fingerprint helpers

The engine loader is the minimal Experiment 4/5/6 file, not the full original notebook. The overflow patch prevents Pandas from coercing enormous interval bitsets into numeric dtype.


In [ ]:

exec(open(BASE / "src" / "deu_exp456_minimal.py", encoding="utf-8").read(), globals())
exec(open(BASE / "src" / "deu_exp456_overflow_patch.py", encoding="utf-8").read(), globals())
exec(open(BASE / "src" / "deu_sr_fingerprint.py", encoding="utf-8").read(), globals())
print("Loaded minimal engine + overflow patch + SR fingerprint helpers.")



## Configuration

Start with `RUN_MODE = "smoke"` to confirm the workflow. Switch to `RUN_MODE = "paper"` for the cap-256 10k-split audit.

The paper-mode defaults use the previously identified cap-256 phase window `ep_lo=9`, `ep_hi=33` and broad gap bins. The notebook then selects the best local contiguous gap-bin window by a fixed score; it does not manually pick the window after seeing the layer profiles.


In [ ]:

RUN_MODE = "smoke"   # "smoke" or "paper"
RUN_FOAM = True       # set False to run controls only
USE_EXISTING_FOAM_VARIABLES = False  # advanced: use rg2/diamonds variables already in memory

if RUN_MODE == "smoke":
    CONTROL_SIZES = (30, 50, 80, 120)
    CONTROL_REPS = 2
    FOAM_KW = dict(
        cap=128,
        seed=101,
        target_basin_splits=1200,
        ep_lo=None,
        ep_hi=None,
        gap_bins=((2,4), (4,6), (6,8), (8,12)),
        n_endpoint_samples_per_bin=1200,
        keep_top_per_bin=30,
        n_profile_bins=21,
    )
elif RUN_MODE == "paper":
    CONTROL_SIZES = (40, 60, 90, 130, 190, 280, 420, 620)
    CONTROL_REPS = 4
    FOAM_KW = dict(
        cap=256,
        seed=101,
        target_basin_splits=10000,
        ep_lo=9,
        ep_hi=33,
        gap_bins=((2,4), (4,6), (6,8), (8,12), (12,16), (16,20), (20,24)),
        n_endpoint_samples_per_bin=25000,
        keep_top_per_bin=200,
        n_profile_bins=21,
    )
else:
    raise ValueError("RUN_MODE must be 'smoke' or 'paper'")

print("RUN_MODE:", RUN_MODE)
print("FOAM_KW:", FOAM_KW)



## Generate flat-SR controls

Controls are fixed-endpoint intervals:

- `chain_1D`
- `minkowski_2p1D`
- `minkowski_3p1D`
- `random_DAG`

Each control produces the same interval fingerprints as the foam: MM dimension, ordering fraction, volume/height scaling, antichain scaling, and normalized layer profiles.


In [ ]:

control_intervals, control_profiles = build_control_fingerprints(
    sizes=CONTROL_SIZES,
    reps_per_size=CONTROL_REPS,
    seed=123,
    n_profile_bins=21,
    random_dag_p=0.025,
)

control_summary = summarize_interval_fingerprints(control_intervals)
display(control_summary[[
    "model", "n_intervals", "h_min", "h_med", "h_max", "r_med", "MM_med",
    "volume_slope", "volume_r2", "mid_layer_slope", "mid_layer_r2",
    "max_layer_slope", "max_layer_r2", "sr_score_2p1D"
]])

control_intervals.to_csv(OUT / f"sr_controls_intervals_{RUN_MODE}.csv", index=False)
control_profiles.to_csv(OUT / f"sr_controls_profiles_{RUN_MODE}.csv", index=False)
control_summary.to_csv(OUT / f"sr_controls_summary_{RUN_MODE}.csv", index=False)



## Generate or load foam diamonds

This is the only heavy section. In `paper` mode it recomputes a cap-256 10k-split phase-window audit, then automatically selects a local SR-like gap-bin window using the predeclared score in `choose_local_gap_window`.

If you have already produced `rg2_cap256` and `multi_diamonds_cap256`/`phase_diamonds_cap256` in the same kernel, set `USE_EXISTING_FOAM_VARIABLES=True` and adapt the variable names in the first branch below.


In [ ]:

foam_result = None
foam_window_label = None

if RUN_FOAM and USE_EXISTING_FOAM_VARIABLES:
    # Adapt these names if your previous notebook used different variables.
    rg2_existing = globals().get("phase_rg2_cap256", globals().get("rg2_cap256", None))
    diamonds_existing = globals().get("phase_diamonds_cap256", globals().get("multi_diamonds_cap256", None))
    binned_existing = globals().get("phase_binned_cap256", globals().get("multi_binned_cap256", None))
    if rg2_existing is None or diamonds_existing is None:
        raise RuntimeError("Could not find existing rg2/diamond variables. Set USE_EXISTING_FOAM_VARIABLES=False or edit this branch.")
    local_existing = choose_local_gap_window(binned_existing) if binned_existing is not None else pd.DataFrame()
    selected_bins = tuple(local_existing.iloc[0]["gap_bin_filter"]) if len(local_existing) else None
    foam_all_intervals, foam_all_profiles = extract_diamond_fingerprints(rg2_existing, diamonds_existing, model="foam_existing_all", n_profile_bins=21)
    foam_window_intervals, foam_window_profiles = extract_diamond_fingerprints(rg2_existing, diamonds_existing, model="foam_existing_SR_window", n_profile_bins=21, gap_bin_filter=selected_bins)
    foam_result = {
        "binned": binned_existing,
        "local_windows": local_existing,
        "selected_gap_bins": selected_bins,
        "foam_all_intervals": foam_all_intervals,
        "foam_all_profiles": foam_all_profiles,
        "foam_window_intervals": foam_window_intervals,
        "foam_window_profiles": foam_window_profiles,
    }
    foam_window_label = "foam_existing_SR_window"

elif RUN_FOAM:
    foam_result = run_foam_sr_sample(**FOAM_KW)
    foam_window_label = f"foam_cap{FOAM_KW['cap']}_SR_window"

else:
    print("RUN_FOAM=False: controls only.")

if foam_result is not None:
    print("Selected gap bins:", foam_result["selected_gap_bins"])
    display(foam_result["local_windows"].head(10))
    display(foam_result["binned"][[
        "gap_bin", "gap_lo", "gap_hi", "n", "h_med", "h_min", "h_max", "V_med", "V_max",
        "mid_layer_med", "max_layer_med", "MM_med", "r_med"
    ]])

    foam_result["foam_all_intervals"].to_csv(OUT / f"sr_foam_all_intervals_{RUN_MODE}.csv", index=False)
    foam_result["foam_all_profiles"].to_csv(OUT / f"sr_foam_all_profiles_{RUN_MODE}.csv", index=False)
    foam_result["foam_window_intervals"].to_csv(OUT / f"sr_foam_window_intervals_{RUN_MODE}.csv", index=False)
    foam_result["foam_window_profiles"].to_csv(OUT / f"sr_foam_window_profiles_{RUN_MODE}.csv", index=False)
    foam_result["binned"].to_csv(OUT / f"sr_foam_binned_{RUN_MODE}.csv", index=False)
    foam_result["local_windows"].to_csv(OUT / f"sr_foam_local_windows_{RUN_MODE}.csv", index=False)



## Combine foam + controls and score SR compatibility

The most important row is the selected foam SR window. A promising result should have:

- `MM_med` near 3,
- `volume_slope` near 3,
- `mid_layer_slope` and/or `max_layer_slope` near 2,
- a low `sr_score_2p1D`.


In [ ]:

if foam_result is not None:
    all_intervals = pd.concat([
        control_intervals,
        foam_result["foam_all_intervals"],
        foam_result["foam_window_intervals"],
    ], ignore_index=True)
    all_profiles = pd.concat([
        control_profiles,
        foam_result["foam_all_profiles"],
        foam_result["foam_window_profiles"],
    ], ignore_index=True)
else:
    all_intervals = control_intervals.copy()
    all_profiles = control_profiles.copy()

sr_summary = summarize_interval_fingerprints(all_intervals)
display(sr_summary[[
    "model", "n_intervals", "h_min", "h_med", "h_max", "r_med", "MM_med",
    "volume_slope", "volume_r2", "mid_layer_slope", "mid_layer_r2",
    "max_layer_slope", "max_layer_r2", "sr_score_2p1D"
]])

sr_summary.to_csv(OUT / f"sr_combined_summary_{RUN_MODE}.csv", index=False)
all_intervals.to_csv(OUT / f"sr_combined_intervals_{RUN_MODE}.csv", index=False)
all_profiles.to_csv(OUT / f"sr_combined_profiles_{RUN_MODE}.csv", index=False)



## Layer-profile distance test

This asks whether the foam's normalized interval layer profile is closest to the $2+1$D Minkowski profile or to some other control.


In [ ]:

avg_profiles = average_profiles(all_profiles)
avg_profiles.to_csv(OUT / f"sr_average_profiles_{RUN_MODE}.csv", index=False)

if foam_window_label is not None and foam_window_label in set(avg_profiles["model"]):
    dist_to_foam = profile_distance_table(avg_profiles, foam_window_label)
    display(dist_to_foam)
    dist_to_foam.to_csv(OUT / f"sr_profile_distances_to_foam_{RUN_MODE}.csv", index=False)

    selected_window_row = None
    if foam_result is not None and len(foam_result.get("local_windows", [])):
        selected_window_row = foam_result["local_windows"].iloc[0]
    interpretation = interpret_sr_fingerprint_v2(sr_summary, dist_to_foam, foam_window_label, selected_window_row)
    print("INTERPRETATION:", interpretation)
    (OUT / f"sr_interpretation_{RUN_MODE}.txt").write_text(interpretation, encoding="utf-8")
else:
    dist_to_foam = pd.DataFrame()
    print("No foam SR window label found; skipping foam profile-distance interpretation.")

if "minkowski_2p1D" in set(avg_profiles["model"]):
    dist_to_minkowski = profile_distance_table(avg_profiles, "minkowski_2p1D")
    display(dist_to_minkowski)
    dist_to_minkowski.to_csv(OUT / f"sr_profile_distances_to_minkowski_2p1D_{RUN_MODE}.csv", index=False)



## Figures

The figures are saved under `figures/` and can be used directly in the manuscript if the audit is promising.


In [ ]:

models_for_plot = [
    "chain_1D",
    "minkowski_2p1D",
    "minkowski_3p1D",
    "random_DAG",
]
if foam_window_label is not None:
    models_for_plot.append(foam_window_label)

plot_layer_profiles(avg_profiles, outfile=str(FIG / f"sr_layer_profiles_{RUN_MODE}.png"), models=models_for_plot)
plot_scaling(all_intervals, outfile=str(FIG / f"sr_volume_scaling_{RUN_MODE}.png"), models=models_for_plot)
plot_antichain_scaling(all_intervals, outfile=str(FIG / f"sr_antichain_scaling_{RUN_MODE}.png"), models=models_for_plot)

print("Saved:")
print(FIG / f"sr_layer_profiles_{RUN_MODE}.png")
print(FIG / f"sr_volume_scaling_{RUN_MODE}.png")
print(FIG / f"sr_antichain_scaling_{RUN_MODE}.png")



## Compact markdown report

This saves a short report so you do not need to paste large tables back into chat. Share the CSVs or the report file instead.


In [ ]:

if foam_window_label is not None and len(dist_to_foam):
    write_sr_report(sr_summary, dist_to_foam, OUT / f"sr_report_{RUN_MODE}.md")
    print((OUT / f"sr_report_{RUN_MODE}.md").read_text(encoding="utf-8")[:4000])
else:
    print("No foam report generated because no foam window was available.")



## How to read failure modes

- **MM near 3 but layer profile not closest to $2+1$D Minkowski:** MM shoulder is not enough; do not claim flat SR structure.
- **Volume slope not near 3:** proper-time/volume law fails; stop before GR.
- **Antichain slope not near 2:** spatial cross-section law fails; the foam may be dimension-3 by ordering fraction only.
- **Foam closest to random-DAG:** the shoulder is likely non-manifoldlike.
- **Foam closest to $3+1$D control:** the phase is not behaving like the triangular primitive expectation.

A pass does not prove GR. It says the triangular foam passes a flat $2+1$D Lorentzian/SR kinematic screen, which is a necessary precursor for any future curvature tests.
